# FaceForensics++ Deepfake Detection Experimental Pipeline
This notebook provides a modular, step-by-step workflow to reproduce the experiments presented in the paper using the official FaceForensics++ Deepfakes subset.

### Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone repository and set working directory
!git clone https://github.com/Deeptrust-us/deepfake_image_video.git /content/deepfake_image_video
%cd /content/deepfake_image_video

# Install requirements
!pip install -r requirements.txt
!apt-get update && apt-get install -y ffmpeg

### Step 2: Download FaceForensics++ Deepfakes Subset
Downloads the complete 1,000 real (youtube) and 1,000 fake (Deepfakes) videos at c23 compression quality.

In [ ]:
# Download full Deepfakes subset
!python scripts/download/download_faceforensics.py data/faceforensics_raw -d Deepfakes -c c23 -t videos -y

# Download corresponding original YouTube sequences
!python scripts/download/download_faceforensics.py data/faceforensics_raw -d original -c c23 -t videos -y

### Step 3: Run Dataset Preprocessing
Runs face extraction, alignment, frequency spectrum calculations, and saves training manifests. Enforces official splits and strict cross-split leakage exclusions.

In [ ]:
# Preprocess dataset (creates 32 uniform frames per video and assigns official train/val/test splits)
!python scripts/preprocess.py --config config/config.yaml --dataset-type faceforensics --videos-dir data/faceforensics_raw

### Step 4: Run Overfitting and System Integrity Verification
Runs standalone overfitting tests on models to guarantee software and network convergence before launching long experiments.

In [ ]:
# Run EER and Metrics unit tests
!python -m unittest tests/test_metrics.py

# Run overfitting test for Xception baseline
!python scripts/overfit_test.py --config config/config.yaml --model xception

# Run overfitting test for Quad-Stream model
!python scripts/overfit_test.py --config config/config.yaml --model quad_stream

### Step 5: Run Xception Baseline Experiments
Trains and evaluates Xception face-only baseline across three seeds (42, 123, 2026).

In [ ]:
# Train and evaluate Xception model baseline over seeds
!python scripts/run_pipeline.py --model xception --epochs 30 --seeds 42 123 2026

### Step 6: Run Quad-Stream Model Experiments
Trains and evaluates the complete Quad-Stream architecture over three seeds (42, 123, 2026).

In [ ]:
# Train and evaluate complete Quad-Stream model over seeds
!python scripts/run_pipeline.py --model quad_stream --epochs 40 --seeds 42 123 2026

### Step 7: Print Consolidated Results & Package Logs
Summarizes the final video-level results and packages logs and checkpoints for archiving.

In [ ]:
# Print final results summary table
!cat results/pipeline_summary.txt

# Archive results, logs and config files for reproducibility
!tar -czf results_archive.tar.gz results/ logs/ config/ checkpoints/